# Test notebook for DPD funcionality

In [1]:
import warnings

warnings.filterwarnings("ignore")

In [2]:
from flowermd.library import PPS

molecules = PPS(num_mols=30, lengths=30)
molecules.coarse_grain(beads={"_A": "c1cc(S)ccc1"})
molecules.bond_length

0.176

## New Random Walk System Class

In [3]:
from flowermd.library import RandomWalk
import unyt as u

ref_length = 0.3438 * u.Unit("nm")
ref_mass = 32.06 * u.Unit("amu")
ref_energy = 1.065 * u.Unit("kJ/mol")
ref_values_dict = {"length": ref_length, "mass": ref_mass, "energy": ref_energy}

system = RandomWalk(
    molecules=molecules,
    density=1.32 * u.Unit("g/cm**3"),
    bond_length=1.4226,
    buffer=0.58,
    base_units=ref_values_dict,
)

(900, 3)


In [4]:
print(system.reference_mass)

32.06 amu


In [5]:
system.box.Lx

np.float64(4.966861)

## FF from GMSO XML file

In [6]:
from flowermd.library.forcefields import Bead_Spring_DPD

In [7]:
system.apply_forcefield(force_field=Bead_Spring_DPD(), r_cut=1.15)

EngineIncompatibilityError: Potential <AtomType _A, expression: A*(-r/r_cut + 1) - γ, id: 125884950803344> is not in the list of accepted_potentials (<PotentialTemplate LennardJonesPotential,
 expression: 4*epsilon*(-sigma**6/r**6 + sigma**12/r**12),
 id: 125884950400928>, <PotentialTemplate HarmonicBondPotential,
 expression: 0.5*k*(r - r_eq)**2,
 id: 125884953198288>, <PotentialTemplate HOOMDFENEWCABondPotential,
 expression: 4*epsilon*(sigma**12/(-delta + r)**12 - sigma**6/(-delta + r)**6) + epsilon - 0.5*k*r0**2*log(1 - (-delta + r)**2/r0**2),
 id: 125891485918912>, <PotentialTemplate HarmonicAnglePotential,
 expression: 0.5*k*(theta - theta_eq)**2,
 id: 125884938364208>, <PotentialTemplate PeriodicTorsionPotential,
 expression: k*(cos(n*phi - phi_eq) + 1),
 id: 125884938364448>, <PotentialTemplate HOOMDPeriodicDihedralPotential,
 expression: 0.5*k*(d*cos(n*phi - phi0) + 1),
 id: 125884938364608>, <PotentialTemplate HarmonicTorsionPotential,
 expression: 0.5*k*(phi - phi_eq)**2,
 i

## Forcefield class

In [ ]:
from flowermd.library import DPD

A = 2000
gamma = 1500
kT = 1.0
r_cut = 1.4226
bond_k = 50000
bond_r0 = 1.4226
dpd_ff = DPD(
    A=A, gamma=gamma, kT=kT, r_cut=r_cut, bond_k=bond_k, bond_r0=bond_r0
)

In [ ]:
from flowermd.library.simulations.dpd_init import DPDInit

sim = DPDInit(
    initial_state=system.hoomd_snapshot,
    forcefield=dpd_ff.hoomd_forces,
    gsd_write_freq=100,
    log_write_freq=100,
    A=A,
    r_cut=r_cut,
    r=1.2,
    N=molecules.n_particles,
    box=system.box.Lx,
    sim_steps_incr=100,
)

In [ ]:
import hoomd

for writer in sim.operations.writers:
    if isinstance(writer, hoomd.write.GSD):
        writer.flush()

In [ ]:
# system.to_gsd("random_walk_test.gsd")